Imports

In [1]:
import os, random
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from spikingjelly.clock_driven import neuron, surrogate, encoding, functional

Dataset

In [2]:
class BinaryImages(Dataset):
    def __init__(self):
        self.samples = []
        self.labels = []

        for fname in os.listdir("zero_imgs"):
            self.samples.append(f"zero_imgs/{fname}")
            self.labels.append(0)

        for fname in os.listdir("one_imgs"):
            self.samples.append(f"one_imgs/{fname}")
            self.labels.append(1)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path = self.samples[idx] 
        label = self.labels[idx] 
        img = Image.open(path).convert("L") #open image as grayscale
        arr = np.array(img, dtype=np.uint8) #convert to numpy array
        x = torch.from_numpy(arr).unsqueeze(0).float() / 255.0 #convert to tensor and normalize
        return x, label

Model

In [3]:
class BasicBinarySNN(nn.Module):
    def __init__(self):
        super().__init__()
        v_th=0.20
        tau=2.0
        input_gain=3.0
        self.input_gain = input_gain
        
        self.flatten = nn.Flatten()
        self.fc1     = nn.Linear(9, 6, bias=False)
        self.lif1    = neuron.LIFNode(tau=tau, v_threshold=v_th, surrogate_function=surrogate.Sigmoid(alpha=4.0), detach_reset=True)
        self.fc2     = nn.Linear(6, 2, bias=False)
        self.lif2   = neuron.LIFNode(tau=tau, v_threshold=v_th, surrogate_function=surrogate.Sigmoid(alpha=4.0), detach_reset=True)

    def forward(self, x):
        x = x * self.input_gain
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.lif1(x)
        x = self.fc2(x)
        x = self.lif2(x)
        return x.float()

Training

In [7]:
def train(model, train_loader, device):
    num_epochs=20
    T=32
    base_lr=1e-3 
    weight_decay=1e-5 
    max_grad_norm=1.0
    
    #training setup
    optimizer = torch.optim.Adam(model.parameters(), lr=base_lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    encoder = encoding.PoissonEncoder()
    criterion = nn.CrossEntropyLoss()

    #training loop
    for epoch in range(num_epochs):
        model.train()

        total_loss = 0 
        total_correct = 0
        total_samples = 0

        for imgs, labels in train_loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            out_counts = 0 #accumulate spike outputs
            for _ in range(T):
                x_t = encoder(imgs) #encode image into spikes
                out_counts = out_counts + model(x_t) #run through SNN and accumulate spikes

            loss = criterion(out_counts, labels)

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()

            total_loss += loss.item() * labels.size(0)
            total_correct += (out_counts.argmax(1) == labels).sum().item()
            total_samples += labels.size(0)

            functional.reset_net(model) #reset spiking neurons for next batch

        avg_loss = total_loss / total_samples
        accuracy = total_correct / total_samples
        print(f"Epoch {epoch+1} Loss: {avg_loss:.4f} Acc: {accuracy:.4f}")
        scheduler.step()

    return model


Main

In [ ]:
batch_size = 1

random.seed(42); torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

dataset = BinaryImages()
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)

model = BasicBinarySNN().to(device)
#intentional overfitting. Only a certain number of combinations of pixels in 3x3 for 0 and 1.
model = train(model, train_loader, device) 

#save model weights
torch.save(model.state_dict(), "binary_snn_weights.pth")

Epoch 1 Loss: 13.7213 Acc: 0.4667
Epoch 2 Loss: 7.4408 Acc: 0.4222
Epoch 3 Loss: 4.0447 Acc: 0.3111
Epoch 4 Loss: 2.7576 Acc: 0.4000
Epoch 5 Loss: 1.7841 Acc: 0.5000
Epoch 6 Loss: 1.0311 Acc: 0.6556
Epoch 7 Loss: 1.0058 Acc: 0.6333
Epoch 8 Loss: 0.7959 Acc: 0.6778
Epoch 9 Loss: 0.3938 Acc: 0.7889
Epoch 10 Loss: 0.2095 Acc: 0.8000
Epoch 11 Loss: 0.1603 Acc: 0.8333
Epoch 12 Loss: 0.0007 Acc: 1.0000
Epoch 13 Loss: 0.0013 Acc: 1.0000
Epoch 14 Loss: 0.0009 Acc: 1.0000
Epoch 15 Loss: 0.0007 Acc: 1.0000
Epoch 16 Loss: 0.0082 Acc: 1.0000
Epoch 17 Loss: 0.0011 Acc: 1.0000
Epoch 18 Loss: 0.0009 Acc: 1.0000
Epoch 19 Loss: 0.0009 Acc: 1.0000
Epoch 20 Loss: 0.0009 Acc: 1.0000
